# E2-B: Data Preparation and Master Table

**Spec sections**: 1.2 (Data Baseline), 2.1–2.3, Chapter 4, Appendix D.8

Build the master table: validate target, apply population filters, assign windows,
reconcile counts against spec 1.2, and produce all step-1 reports.

In [ ]:
%fs ls /Volumes/mis_automation/datascience_pbg/pbg_cc_ascore/output

In [ ]:
import sys, os, shutil, tempfile

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import common

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
id_col     = cfg["columns"]["id"]
date_col   = cfg["columns"]["date"]
target_col = cfg["columns"]["target"]
cat        = cfg["databricks"]["catalog"]
sch        = cfg["databricks"]["schema"]

ARTIFACTS = common.artifacts_path(cfg)
REPORTS   = os.path.join(ARTIFACTS, "reports")

# Create UC schema, volume and reports directory
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {cat}.{sch}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {cat}.{sch}.{cfg['databricks']['volume']}")
os.makedirs(REPORTS, exist_ok=True)

# Helper: Excel writes need seek() which UC Volumes don't support.
# Write to local temp, then copy.
_TMP = tempfile.mkdtemp(prefix="e2b_")

def save_excel(df, filename):
    tmp = os.path.join(_TMP, filename)
    df.to_excel(tmp, index=False)
    shutil.copy2(tmp, os.path.join(REPORTS, filename))
    print(f"Saved: {os.path.join(REPORTS, filename)}")

def save_fig(fig, filename):
    tmp = os.path.join(_TMP, filename)
    fig.savefig(tmp, dpi=150)
    shutil.copy2(tmp, os.path.join(REPORTS, filename))
    print(f"Saved: {os.path.join(REPORTS, filename)}")

print(f"Project: {cfg['project']}  |  Config hash: {common.config_hash(cfg)[:12]}")
print(f"Source table: {cfg['data']['raw_snapshot_table']}")
print(f"Reports dir:  {REPORTS}")

In [ ]:
# Read source table (Appendix F: Spark -> .toPandas())
raw = spark.table(cfg["data"]["raw_snapshot_table"]).toPandas()
print(f"Source shape: {raw.shape}")

# Validate target column (spec 2.2): must only take 1, 0, -1, -2 with no nulls
target_vals = set(raw[target_col].unique())
allowed = {1, 0, -1, -2}
print(f"Target values found: {sorted(target_vals)}")
print(f"Nulls in target: {raw[target_col].isna().sum()}")
assert raw[target_col].isna().sum() == 0, "target has nulls — STOP"
assert target_vals <= allowed, f"target has unexpected values {target_vals - allowed} — STOP"
print("PASS: target only takes 1, 0, -1, -2 with no nulls.")

# Check app_id uniqueness (spec 4.3)
n_unique = raw[id_col].nunique()
if not raw[id_col].is_unique:
    n_dup = len(raw) - n_unique
    print(f"WARNING: {id_col} is not unique ({n_unique:,} unique of {len(raw):,}; "
          f"{n_dup:,} extra rows). Table is at card level.")
    # Exact duplicate rows (same CRNNO + CREDITCARDNO)
    exact_dup = raw.duplicated(subset=[id_col, "CREDITCARDNO"], keep="first")
    if exact_dup.sum() > 0:
        print(f"  Dropping {exact_dup.sum()} exact duplicate rows.")
        raw = raw[~exact_dup].reset_index(drop=True)
        print(f"  New shape: {raw.shape}")
else:
    print(f"PASS: no duplicate {id_col} ({len(raw):,} rows).")

In [ ]:
# Print file-type distribution before any filtering (spec 2.3 disclosure)
ft_col = cfg["file_type"]["source_column"]
ft_norm = common._norm(raw[ft_col])
print(f"File-type column: {ft_col}")
print(f"Value counts (normalised, before filtering):")
vc = ft_norm.value_counts(dropna=False)
for v, c in vc.items():
    print(f"  {str(v):20s}  {c:>6,}  ({c/len(raw):.1%})")
print(f"  {'TOTAL':20s}  {len(raw):>6,}")

In [ ]:
# Apply population filters: sample_status, exclusion_reason, waterfall
master, waterfall = common.apply_population_filters(raw, cfg)
print("Population filters applied.")
print(f"Waterfall rows: {len(waterfall)}")
display(waterfall)

In [ ]:
# Assign window (DEV/OOT/FWD/STRESS), app_month, app_quarter
master = common.assign_window(master, cfg)
print("Window distribution:")
print(master["window"].value_counts(dropna=False).to_string())
print(f"\nRows outside any window: {master['window'].isna().sum()}")

In [ ]:
# Reconcile GOOD+BAD and Bads against config.expected_counts (spec 1.2, 4.3)
# For reconciliation we count on the FILTERED master where sample_status is
# GOOD or BAD (i.e. not EXCLUDED, not EXCLUDED_PERF, not INDETERMINATE, not IMMATURE).
labelled = master[master["sample_status"].isin(cfg["labelled_statuses"])].copy()

def _counts(df):
    return {"records": len(df), "bads": int((df[target_col] == 1).sum())}

# Window boundaries for DEV_EARLY / DEV_LATE (from config.backtest)
bt = cfg["backtest"]
dev_early_end = bt["train"][1]   # 2024-07-01
dev_data = labelled[labelled["window"] == "DEV"]
d = pd.to_datetime(dev_data[date_col])

actual = {
    "ALL":       _counts(labelled[labelled["window"].isin(["DEV", "OOT"])]),
    "DEV":       _counts(dev_data),
    "DEV_EARLY": _counts(dev_data[d < dev_early_end]),
    "DEV_LATE":  _counts(dev_data[d >= dev_early_end]),
    "OOT":       _counts(labelled[labelled["window"] == "OOT"]),
}

expected = cfg["expected_counts"]
all_pass = True
print(f"{'Segment':<12s}  {'Exp Recs':>8s}  {'Act Recs':>8s}  {'Exp Bads':>8s}  {'Act Bads':>8s}  Result")
print("-" * 72)
for seg in ["ALL", "DEV", "DEV_EARLY", "DEV_LATE", "OOT"]:
    er, eb = expected[seg]["records"], expected[seg]["bads"]
    ar, ab = actual[seg]["records"], actual[seg]["bads"]
    ok = (er == ar) and (eb == ab)
    tag = "PASS" if ok else "FAIL"
    if not ok:
        all_pass = False
    print(f"{seg:<12s}  {er:>8,}  {ar:>8,}  {eb:>8,}  {ab:>8,}  {tag}")

if not all_pass:
    print("\n" + "!"*60)
    print("RECONCILIATION FAILED — counts do not match spec 1.2.")
    print("Per spec 4.3 and task instruction: STOP and report.")
    print("Do NOT adjust filters to make numbers match.")
    print("!"*60)
    # Write difference report
    for seg in ["ALL", "DEV", "DEV_EARLY", "DEV_LATE", "OOT"]:
        er, eb = expected[seg]["records"], expected[seg]["bads"]
        ar, ab = actual[seg]["records"], actual[seg]["bads"]
        if er != ar or eb != ab:
            print(f"  {seg}: records {ar-er:+d}, bads {ab-eb:+d}")
else:
    print("\nAll counts reconcile with spec 1.2.")

In [ ]:
labelled.aecb_income_age_months.isna().mean()

In [ ]:
# Save the exclusion waterfall to Excel (spec 4.2)
save_excel(waterfall, "01_exclusion_waterfall.xlsx")
display(waterfall)

In [ ]:
# Monthly vintage table by app_month (spec 4.1 task 1.4)
# Count per sample_status, plus Bad Rate and Indeterminate share
vintage_rows = []
for month, grp in master.groupby("app_month", observed=True):
    sc = grp["sample_status"].value_counts()
    total = len(grp)
    good = int(sc.get("GOOD", 0))
    bad  = int(sc.get("BAD", 0))
    indet = int(sc.get("INDETERMINATE", 0))
    immature = int(sc.get("IMMATURE", 0))
    excl = int(sc.get("EXCLUDED", 0))
    excl_perf = int(sc.get("EXCLUDED_PERF", 0))
    gb = good + bad
    br = bad / gb if gb > 0 else None
    matured = good + bad + indet
    indet_share = indet / matured if matured > 0 else None
    vintage_rows.append({
        "app_month": str(month), "total": total,
        "GOOD": good, "BAD": bad, "INDETERMINATE": indet,
        "IMMATURE": immature, "EXCLUDED": excl, "EXCLUDED_PERF": excl_perf,
        "bad_rate": br, "indeterminate_share": indet_share,
    })
vintage_df = pd.DataFrame(vintage_rows)
save_excel(vintage_df, "01_vintage_monthly.xlsx")
print(f"Months: {len(vintage_df)}")
display(vintage_df.head(10))

In [ ]:
# Cross-tab of ACTIVE_REPORTING against target in DEV (spec 2.3 disclosure)
# Shows count and Bad Rate of records removed by the ACTIVE_REPORTING rule
dev_all = master[master["window"] == "DEV"].copy()
ar_col = "ACTIVE_REPORTING"
ar_norm = common._norm(dev_all[ar_col])
dev_all["AR_norm"] = ar_norm

# Only mature records (target in 0,1) for Bad Rate
dev_mature = dev_all[dev_all[target_col].isin([0, 1])]
xtab = dev_mature.groupby("AR_norm", dropna=False).agg(
    count=(target_col, "size"),
    bads=(target_col, "sum"),
).reset_index()
xtab["bad_rate"] = xtab["bads"] / xtab["count"]
print("ACTIVE_REPORTING x target cross-tab (DEV, mature records):")
display(xtab)

save_excel(xtab, "01_active_reporting_xtab.xlsx")

In [ ]:
# Missing-rate and special-value share by app_quarter for every FEATURE column
# (spec 4.1 task 1.7)
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
feature_cols = fdict.loc[fdict["role"] == "FEATURE", "feature"].tolist()

# Parse special codes from feature_dict
def parse_specials(s):
    if pd.isna(s) or s == "" or s == "nan":
        return []
    try:
        return [float(x) for x in s.strip("[]").split(",") if x.strip()]
    except Exception:
        return []

special_map = dict(zip(fdict["feature"], fdict["special_codes"].apply(parse_specials)))

quarters = sorted(master["app_quarter"].dropna().unique())
missing_rows = []
for col in feature_cols:
    if col not in master.columns:
        continue
    specials = special_map.get(col, [])
    for q in quarters:
        subset = master.loc[master["app_quarter"] == q, col]
        n = len(subset)
        if n == 0:
            continue
        miss = float(subset.isna().sum()) / n
        if specials:
            num = pd.to_numeric(subset, errors="coerce")
            sp_share = float(num.isin(specials).sum()) / n
        else:
            sp_share = 0.0
        missing_rows.append({"feature": col, "app_quarter": str(q),
                             "missing_rate": round(miss, 4),
                             "special_value_share": round(sp_share, 4)})

missing_df = pd.DataFrame(missing_rows)
save_excel(missing_df, "01_missing_by_quarter.xlsx")
print(f"({len(missing_df)} rows, {len(feature_cols)} features)")

In [ ]:
# Monthly Bad Rate chart with policy-change dates overlaid (spec 4.1 task 1.6)
# Only use GOOD+BAD records (labelled)
labelled_w_month = labelled.copy()
labelled_w_month["app_month_dt"] = pd.to_datetime(
    labelled_w_month["app_month"].astype(str)
)
monthly = labelled_w_month.groupby("app_month_dt").agg(
    count=(target_col, "size"), bads=(target_col, "sum")
).reset_index()
monthly["bad_rate"] = monthly["bads"] / monthly["count"]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly["app_month_dt"], monthly["bad_rate"], marker="o", linewidth=1.5)
ax.set_ylabel("Bad Rate")
ax.set_title("Monthly Bad Rate (GOOD+BAD) with Policy Changes")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
fig.autofmt_xdate()

# Overlay policy changes if available
policy_path = os.path.join(PROJECT_ROOT, "config", "policy_changes.csv")
try:
    policy = pd.read_csv(policy_path)
    if len(policy) > 0 and "date" in policy.columns:
        for _, row in policy.iterrows():
            dt = pd.to_datetime(row["date"])
            ax.axvline(dt, color="red", linestyle="--", alpha=0.7)
            ax.text(dt, ax.get_ylim()[1] * 0.95,
                    row.get("description", "")[:30],
                    rotation=90, fontsize=7, va="top", color="red")
        print(f"Overlaid {len(policy)} policy-change dates.")
    else:
        print("policy_changes.csv has no rows or no 'date' column — overlay skipped.")
except FileNotFoundError:
    print("config/policy_changes.csv not found — overlay skipped.")

ax.grid(True, alpha=0.3)
fig.tight_layout()
save_fig(fig, "01_badrate_policy.png")
plt.show()

In [ ]:
# Vintage curves (spec 4.1 task 1.5): need month-on-book delinquency columns.
# Check if they exist in the source data.
mob_candidates = [c for c in master.columns if "mob" in c.lower() and "dpd" in c.lower()]
bucket_candidates = [c for c in master.columns if "bucket" in c.lower() and c != target_col]

if mob_candidates or bucket_candidates:
    print(f"Found potential MOB/bucket columns: {mob_candidates + bucket_candidates}")
    print("Vintage curves can be produced in a follow-up.")
else:
    print("No month-on-book delinquency columns found in the source table.")
    print("Vintage curves (spec 4.1 step 1.5) SKIPPED.")
    # Record in open_questions.md
    oq_path = os.path.join(REPORTS, "open_questions.md")
    with open(oq_path, "a") as f:
        f.write("\n## Vintage curves (spec 4.1 step 1.5)\n")
        f.write("Month-on-book delinquency columns are not available in the source table. ")
        f.write("Vintage curves cannot be produced. Please provide MOB-level delinquency data.\n")
    print(f"Recorded in: {oq_path}")

In [ ]:
# Save the master table as Delta (Appendix F)
# Convert Period columns to strings for Delta compatibility
master_out = master.copy()
master_out["app_month"] = master_out["app_month"].astype(str)
master_out["app_quarter"] = master_out["app_quarter"].astype(str)

master_table = f"{cat}.{sch}.acard_master"
sdf_master = spark.createDataFrame(master_out)
sdf_master.write.format("delta").mode("overwrite").saveAsTable(master_table)
print(f"Master table saved: {master_table}")
print(f"Rows: {sdf_master.count():,}  |  Columns: {len(sdf_master.columns)}")

In [ ]:
# Spec 4.3 acceptance criteria
print("=" * 60)
print("ACCEPTANCE CRITERIA (spec 4.3)")
print("=" * 60)

results = []

# 1. No exact-duplicate rows
check1 = not master[[id_col, "CREDITCARDNO"]].duplicated().any()
results.append(("No exact-duplicate rows (CRNNO+CREDITCARDNO)", check1))

# 1b. CRNNO uniqueness (multiple cards per customer expected)
n_crn = master[id_col].nunique()
results.append((f"{id_col} uniqueness ({n_crn:,} unique of {len(master):,})", master[id_col].is_unique))

# 2. Each row has exactly one sample_status
results.append(("Each row has exactly one sample_status", master["sample_status"].notna().all()))

# 3. target takes only 1,0,-1,-2 with no nulls
check3 = (set(master[target_col].unique()) <= {1, 0, -1, -2}) and (master[target_col].isna().sum() == 0)
results.append(("target only {1,0,-1,-2}, no nulls", check3))

# 4. target identical to source (after exact-dedup)
src = spark.table(cfg["data"]["raw_snapshot_table"]).toPandas()
src = src.drop_duplicates(subset=[id_col, "CREDITCARDNO"], keep="first")
check4 = (master[target_col].values == src[target_col].values).all()
results.append(("target identical to source table", check4))

# 5. Outside EXCLUDED/EXCLUDED_PERF, sample_status matches target
non_excl = master[~master["sample_status"].isin(["EXCLUDED", "EXCLUDED_PERF"])]
target_to_status = {v: k for k, v in cfg["target_codes"].items()}
expected_st = non_excl[target_col].map(target_to_status)
results.append(("sample_status matches target (non-excluded)", (non_excl["sample_status"].values == expected_st.values).all()))

# 6. Count reconciliation
results.append(("Counts reconcile with spec 1.2", all_pass))

# 7. Waterfall produced with rule order
results.append(("Waterfall produced with rule order", len(waterfall) >= 12))

# 8. No WATCH/FLAG/KEY/PERFORMANCE in FEATURE list
fdict_chk = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
non_feat = set(fdict_chk.loc[fdict_chk["role"].isin(["KEY","PERFORMANCE","FLAG","WATCH"]), "feature"])
bad_feat = [c for c in fdict_chk.loc[fdict_chk["role"]=="FEATURE", "feature"] if c in non_feat]
results.append(("No WATCH/FLAG/KEY/PERF in FEATURE list", len(bad_feat) == 0))

# 9. Reports produced
report_files = [
    "01_exclusion_waterfall.xlsx", "01_vintage_monthly.xlsx",
    "01_badrate_policy.png", "01_missing_by_quarter.xlsx",
    "01_active_reporting_xtab.xlsx",
]
results.append(("All spec 4.2 reports produced",
    all(os.path.exists(os.path.join(REPORTS, f)) for f in report_files)))

# 10. Master table saved
try:
    mt_count = spark.table(f"{cat}.{sch}.acard_master").count()
    results.append((f"Master Delta table saved ({mt_count:,} rows)", mt_count == len(master)))
except Exception:
    results.append(("Master Delta table saved", False))

for desc, passed in results:
    print(f"  [{'PASS' if passed else 'FAIL'}]  {desc}")

n_pass = sum(1 for _, p in results if p)
n_fail = sum(1 for _, p in results if not p)
print(f"\n{n_pass} PASS, {n_fail} FAIL out of {len(results)} checks.")
if n_fail > 0:
    print("\nFailed checks require model owner review before proceeding.")
else:
    print("\nE2-B DATA PREPARATION: ALL ACCEPTANCE CRITERIA PASSED")

# Cleanup temp dir
shutil.rmtree(_TMP, ignore_errors=True)